# LangChain: LCEL & Chains & Parsers

## Outline
* فلسفه LCEL — pipe operator
* ساخت chain ساده
* Chain با output parser
* Sequential chains
* Parallel chains با RunnableParallel
* Router chain با Pydantic


## نصب

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-5.2", model_provider="openai", temperature=0)


## ۱. فلسفه LCEL

LCEL (LangChain Expression Language)



In [12]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ساده‌ترین chain
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}"),
])

# pipe operator
chain = prompt | llm | StrOutputParser()

result = chain.invoke("پایتخت ایران کجاست؟")
print(result)


پایتخت ایران **تهران** است.


In [13]:
# streaming با chain
print("Streaming output:")
for chunk in chain.stream({"input": "3 جمله پیرامون AI بگو"}):
    print(chunk, end="", flush=True)
print()


Streaming output:
هوش مصنوعی (AI) به سیستم‌هایی گفته می‌شود که می‌توانند کارهایی مثل یادگیری از داده‌ها، تشخیص الگوها و تصمیم‌گیری را انجام دهند.  
امروزه AI در حوزه‌هایی مانند پزشکی، آموزش، حمل‌ونقل و تولید محتوا کاربردهای گسترده‌ای پیدا کرده است.  
با وجود مزایای زیاد، استفاده از هوش مصنوعی نیازمند توجه جدی به مسائل اخلاقی مثل حریم خصوصی، سوگیری و شفافیت است.


In [16]:
# batch — چند request موازی
results = chain.batch([
    {"input": "Capital of France?"},
    {"input": "Capital of Germany?"},
    {"input": "Capital of Japan?"},
    {"input": "Capital of Iran?"},
])
for r in results:
    print(r)


Paris.
Berlin.
Tokyo.
Tehran.


## ۲. Output Parsers

In [50]:
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from pydantic import BaseModel, Field

llm_gema = init_chat_model("gemma3:4b", model_provider="ollama", temperature=0)

# JsonOutputParser — خروجی JSON
json_prompt = ChatPromptTemplate.from_messages([
    ("system", "Return a 'answer' (Yes/No) and 'confidence' (0-1) keys only. No markdown."),
    ("human", "{question}"),
])

json_chain = json_prompt | llm_gema | JsonOutputParser()
result = json_chain.invoke({"question": "Is the earth round?"})
print(result)
#print(f"Answer: {result['answer']}, Confidence: {result['confidence']}")


{'answer': 'Yes', 'confidence': 1}


In [41]:
result["confidence"]

0.99

In [52]:
# Structured output با Pydantic — روش توصیه‌شده
class ReviewAnalysis(BaseModel):
    """تحلیل احساسات یک نظر"""
    is_product_review: bool = Field(description="True/False, is the review is related to product")
    sentiment: str = Field(description="POSITIVE, NEGATIVE, NEUTRAL or None(if it is not product review)")
    score: float = Field(description="Score from 0 to 1 or -1(if it is not product review)")
    key_points: list[str] = Field(description="Main points from the review, or empty (if is not a review)")

structured_llm = llm.with_structured_output(ReviewAnalysis)

review_prompt = ChatPromptTemplate.from_messages([
    ("system", "Analyze the sentiment of the given review. "),
    ("human", "{review}"),
])

review_chain = review_prompt | structured_llm

result = review_chain.invoke({
    "review": "My name is Gholam"
})
print(f"is review: {result.is_product_review}")
print(f"Sentiment: {result.sentiment}")
print(f"Score: {result.score}")
print(f"Key points: {result.key_points}")


is review: False
Sentiment: None
Score: -1.0
Key points: []


## ۳. Sequential Chains 

In [ ]:

# Chain 1: خلاصه
summarize_prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize the following text in one sentence."),
    ("human", "{text}"),
])
summarize_chain = summarize_prompt | llm | StrOutputParser()

# Chain 2: ترجمه خلاصه
translate_prompt = ChatPromptTemplate.from_messages([
    ("system", "Translate the following text to {lang}. \
                Return only the translated text and nothing else.\
                Do not add explanations, notes, or extra words."),
    ("human", "{text}"),
])
translate_chain = translate_prompt | llm | StrOutputParser()

# ترکیب دو chain
sequential_chain = (
    summarize_chain  | RunnableLambda(lambda x: {"text": x, "lang":"Persian"})|  translate_chain
)

long_text = """
LangChain is a framework for developing applications powered by large language models.
It provides tools and abstractions to improve the customization, accuracy, and relevancy 
of the information the models generate. It includes APIs and integrations for a wide 
variety of components, including vector stores, document loaders, and output parsers.
"""

result = sequential_chain.invoke({"text": long_text})
print("خلاصه فارسی:")
print(result)


In [ ]:
# Sequential chain: خروجی یک chain ورودی بعدی می‌شه
# قدیمی: SimpleSequentialChain / SequentialChain
# جدید: چند chain با | و RunnableLambda

from langchain_core.runnables import RunnableLambda

# Chain 1: خلاصه
summarize_prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize the following text in one sentence."),
    ("human", "{text}"),
])
summarize_chain = summarize_prompt | llm | StrOutputParser()

# Chain 2: ترجمه خلاصه
translate_prompt = ChatPromptTemplate.from_messages([
    ("system", "Translate the following text to Persian. \
                Return only the translated text and nothing else.\
                Do not add explanations, notes, or extra words."),
    ("human", "{text}"),
])
translate_chain = translate_prompt | llm | StrOutputParser()

# ترکیب دو chain
sequential_chain = (
    summarize_chain 
    | RunnableLambda(lambda x: {"text": x})  # خروجی chain اول → ورودی chain دوم
    | translate_chain
)

long_text = """
LangChain is a framework for developing applications powered by large language models.
It provides tools and abstractions to improve the customization, accuracy, and relevancy 
of the information the models generate. It includes APIs and integrations for a wide 
variety of components, including vector stores, document loaders, and output parsers.
"""

result = sequential_chain.invoke({"text": long_text})
print("خلاصه فارسی:")
print(result)


In [ ]:

from langchain_core.runnables import RunnableLambda

# Chain 1: خلاصه
summarize_prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize the following text in one sentence."),
    ("human", "{text}"),
])
summarize_chain = summarize_prompt | llm | StrOutputParser()

# Chain 2: ترجمه خلاصه
translate_prompt = ChatPromptTemplate.from_messages([
    ("system", "Translate the following text to {lang}. \
                Return only the translated text and nothing else.\
                Do not add explanations, notes, or extra words."),
    ("human", "{text}"),
])
translate_chain = translate_prompt | llm | StrOutputParser()

# ترکیب دو chain
sequential_chain = (
    summarize_chain 
    | RunnableLambda(lambda x: {"text": x, "lang":"Persian"})  # خروجی chain اول → ورودی chain دوم
    | translate_chain
)

long_text = """
LangChain is a framework for developing applications powered by large language models.
It provides tools and abstractions to improve the customization, accuracy, and relevancy 
of the information the models generate. It includes APIs and integrations for a wide 
variety of components, including vector stores, document loaders, and output parsers.
"""

result = sequential_chain.invoke({"text": long_text})
print("خلاصه فارسی:")
print(result)


## ۴. Parallel Chains 

In [ ]:
from langchain_core.runnables import RunnableParallel

# Parallel: چند chain همزمان اجرا می‌شن
pros_prompt = ChatPromptTemplate.from_messages([
    ("system", "List 3 pros of the given product in bullet points."),
    ("human", "{product}"),
])

cons_prompt = ChatPromptTemplate.from_messages([
    ("system", "List 3 cons of the given product in bullet points."),
    ("human", "{product}"),
])

summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "Write a one-line summary of the given product."),
    ("human", "{product}"),
])

# هر سه chain موازی اجرا می‌شن
parallel_chain = RunnableParallel(
    pros=(pros_prompt | llm | StrOutputParser()),
    cons=(cons_prompt | llm | StrOutputParser()),
    summary=(summary_prompt | llm | StrOutputParser()),
)

result = parallel_chain.invoke({"product": "iPhone 15 Pro"})
print("=== Summary ===")
print(result["summary"])
print("\n=== Pros ===")
print(result["pros"])
print("\n=== Cons ===")
print(result["cons"])


In [ ]:
import time

# ====== 3 سوال که پاسخشان دقیقاً یک کلمه است ======

# سوال 1: بله/خیر
q1_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer with ONLY ONE WORD, just name of a city."),
    ("human", "What is the Capital of Iran?")
])

# سوال 2: یک عدد
q2_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer with ONLY ONE WORD (a number like 'two' or 'five')."),
    ("human", "How many legs does a dog have?")
])

# سوال 3: یک اسم ساده
q3_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer with ONLY ONE WORD, just the name."),
    ("human", "What color is an apple?")
])

chain1 = q1_prompt | llm | StrOutputParser()
chain2 = q2_prompt | llm | StrOutputParser()
chain3 = q3_prompt | llm | StrOutputParser()

# ====== 1. اجرای عادی 3 بار پشت سر هم ======
start_seq = time.time()

r1 = chain1.invoke({})
r2 = chain2.invoke({})
r3 = chain3.invoke({})

end_seq = time.time()

print("=== حالت عادی (Sequential) ===")
print(f"پاسخ‌ها: '{r1}', '{r2}', '{r3}'")
print(f"زمان: {end_seq - start_seq:.3f} ثانیه\n")

# ====== 2. اجرای موازی ======
start_par = time.time()

parallel_chain = RunnableParallel(
    ans1=chain1,
    ans2=chain2,
    ans3=chain3,
)

results = parallel_chain.invoke({})
end_par = time.time()

print("=== حالت موازی (Parallel) ===")
print(f"پاسخ‌ها: '{results['ans1']}', '{results['ans2']}', '{results['ans3']}'")
print(f"زمان: {end_par - start_par:.3f} ثانیه")
print(f"\n⚡ نسبت بهبود: {(end_seq - start_seq) / (end_par - start_par):.2f}x")

## ۵. Router Chain — جایگزین RouterChain

In [ ]:
from pydantic import BaseModel
from typing import Literal

# Router: بر اساس موضوع، به chain مناسب route می‌کند
class RouteQuery(BaseModel):
    """تشخیص موضوع سوال"""
    topic: Literal["math", "history", "science", "other"]

router_llm = llm.with_structured_output(RouteQuery)

# Chain‌های تخصصی
math_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a math expert. Answer math questions step by step."),
    ("human", "{question}"),
])

history_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a history expert. Provide historical context."),
    ("human", "{question}"),
])

science_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a science expert. Explain scientific concepts clearly."),
    ("human", "{question}"),
])

general_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{question}"),
])

def route(input_dict):
    """تشخیص موضوع و انتخاب chain مناسب"""
    route_result = router_llm.invoke(input_dict["question"])
    topic = route_result.topic
    print(f"→ Routed to: {topic}")    
    chains = {
        "math": math_prompt | llm | StrOutputParser(),
        "history": history_prompt | llm | StrOutputParser(),
        "science": science_prompt | llm | StrOutputParser(),
        "other": general_prompt | llm | StrOutputParser(),
    }
    
    return chains[topic].invoke(input_dict)

# تست router
questions = [
    "What is the derivative of x^2?",
    "When did World War II end?",
    "What is the best programming language?",
    "How does photosynthesis work?",
]

for q in questions:
    print(f"\nQ: {q}")
    print(f"A: {route({'question': q})[:100]}...")
